# Esercizio 1 — Confronto tra metriche automatiche e valutazione umana
In base a quanto appreso durante il corso, si richiede di confrontare tre varianti/modelli (`Base`, `SFT`, `RLHF`) mediante metriche automatiche e valut umana.
---
## 1. Generazione delle risposte
Generare risposte utilizzando il modello GPT-2 tramite le 3 versioni a partire da almeno 3 prompt appartenenti a categorie differenti, ad esempio:
- Salute
- Educazione
- Coding

Per ogni prompt salvare:
- prompt originale
- risposta del modello Base
- risposta del SFT
- risposta del modello RLHF
---
## 2. Valutazione automatica
Calcolare per ciascun modello le seguenti metriche:
- BLEU
- ROUGE-1
- BERT
Utilizzare un testo di riferimento (`reference`) per il confronto con le risposte generate.
---
## 3. Confronto delle metriche
Confrontare i risultati ottenuti dai tre modelli.
Per ciascun modello riportare almeno:

| Modello |cun modello riportare almeno:
| Modello | BLEU | ROUGE-1 | BERTScore |
|----------|------|----------|------------|
| Base     |      |          |            |
| SFT      |      |          |            |
| RLHF     |      |          |            |


In [1]:
### Scrivi la tua soluzione

import json
from pathlib import Path
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from bert_score import score as bert_score

set_seed(42)
device = torch.device('cpu')
cartella_risultati = Path('risultati_parte2')
cartella_risultati.mkdir(exist_ok=True)

MODEL_NAME = 'distilgpt2'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
model_base = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model_sft = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model_rlhf = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
modelli = {'Base': model_base, 'SFT': model_sft, 'RLHF': model_rlhf}
for modello in modelli.values():
    modello.to(device)
    modello.eval()

# Come nell'esercitazione: SFT e RLHF sono simulati con gli stessi pesi.
print('Base, SFT e RLHF: tre copie di DistilGPT-2, senza nuovo training.')

esempi = [
    {
        'Categoria': 'Educazione',
        'Prompt': 'What is reinforcement learning?',
        'Reference': 'Reinforcement learning is a type of ML where an agent learns by interacting with an environment and receiving rewards.'
    },
    {
        'Categoria': 'Coding',
        'Prompt': 'How do I read a CSV file in Python?',
        'Reference': """Use the pandas library for easy CSV handling:

```python
import pandas as pd
df = pd.read_csv('file.csv')
print(df.head())
```

Alternatively, use the built-in csv module for more control."""
    },
    {
        'Categoria': 'Customer care',
        'Prompt': 'How do I track my order?',
        'Reference': 'You can track your order by logging into your account. Go to My Orders and enter your order number.'
    }
]

def generate_response(model, tokenizer, prompt, max_new_tokens=50):
    inputs = tokenizer(prompt, return_tensors='pt')
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            min_new_tokens=10,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            top_k=40,
            repetition_penalty=2.0,
            no_repeat_ngram_size=2,
            pad_token_id=tokenizer.eos_token_id
        )
    # Decodifico solo i nuovi token, senza includere la domanda.
    return tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

def calculate_bleu(reference, candidate):
    smoothing = SmoothingFunction().method1
    return sentence_bleu(
        [reference.lower().split()], candidate.lower().split(),
        smoothing_function=smoothing
    )

def calculate_rouge(reference, candidate):
    scorer = rouge_scorer.RougeScorer(['rouge1'], use_stemmer=True)
    return scorer.score(reference, candidate)['rouge1'].fmeasure

risultati = []
for esempio in esempi:
    prompt = 'Question: ' + esempio['Prompt'] + ' Answer:'
    for nome, modello in modelli.items():
        risposta = generate_response(modello, tokenizer, prompt)
        risultati.append({
            **esempio,
            'Modello': nome,
            'Risposta': risposta,
            'BLEU': float(calculate_bleu(esempio['Reference'], risposta)),
            'ROUGE-1': float(calculate_rouge(esempio['Reference'], risposta))
        })
        print(f"\n{esempio['Categoria']} - {nome}\nPrompt: {esempio['Prompt']}\nRisposta: {risposta}")

candidati = [r['Risposta'] for r in risultati]
riferimenti = [r['Reference'] for r in risultati]
P, R, F1 = bert_score(
    candidati, riferimenti, lang='en', device='cpu', batch_size=1, verbose=False
)
for risultato, f1 in zip(risultati, F1):
    risultato['BERTScore'] = f1.item()

df_risultati = pd.DataFrame(risultati)
df_medie = df_risultati.groupby('Modello', sort=False)[['BLEU', 'ROUGE-1', 'BERTScore']].mean()
print('\nMetriche per risposta:')
print(df_risultati[['Categoria', 'Modello', 'BLEU', 'ROUGE-1', 'BERTScore']].to_string(index=False))
print('\nMedie per modello:')
print(df_medie.to_string())
df_risultati.to_csv(cartella_risultati / 'risposte_metriche.csv', index=False)
df_medie.to_csv(cartella_risultati / 'confronto_metriche.csv')
with open(cartella_risultati / 'risposte.json', 'w', encoding='utf-8') as file:
    json.dump(risultati, file, ensure_ascii=False, indent=2)

criteri = ['Cortesia', 'Informatività', 'Risoluzione', 'Chiarezza', 'Sicurezza']
rubrica = pd.DataFrame({
    'Criterio': criteri,
    'Descrizione': [
        'Tono educato e professionale',
        'Fornisce informazioni utili',
        'Risolve il problema posto',
        'Linguaggio chiaro e comprensibile',
        'Non fornisce informazioni dannose o false'
    ],
    'Scala': ['1-5'] * 5
})
print('\nRubrica: 1 = pessimo, 3 = accettabile, 5 = eccellente')
print(rubrica.to_string(index=False))

# Punteggi assegnati dopo la lettura delle nove risposte con seed 42.
giudizi = [
    ([3, 1, 1, 2, 3], "Non definisce il reinforcement learning; discorso generico e interrotto."),
    ([3, 1, 1, 2, 3], "Parla di esperienza e di un post, senza agente, ambiente o reward."),
    ([3, 2, 1, 2, 2], "Richiama l'apprendimento, ma non spiega il ruolo delle ricompense."),
    ([3, 1, 1, 2, 2], "Propone SQL Server, JSON e MySQL: non legge un CSV con Python."),
    ([3, 1, 1, 2, 2], "Risposta negativa e fuori tema su JSON/XML; manca un esempio CSV."),
    ([3, 1, 1, 2, 3], "Aneddoto personale e array, senza una procedura per il CSV."),
    ([2, 1, 1, 1, 2], "Confusione fra ordine e acquisto; nessuna istruzione di tracking."),
    ([3, 1, 1, 3, 2], "Testo leggibile, ma parla di carrello e Facebook invece del tracking."),
    ([3, 2, 1, 2, 2], "Accenna a pacchi e UPS, ma non indica dove verificare il tracking.")
]
valutazione = df_risultati[['Categoria', 'Prompt', 'Modello', 'Risposta']].copy()
for criterio in criteri:
    valutazione[criterio] = None
valutazione['Commento'] = ''
for i, (punteggi, commento) in enumerate(giudizi):
    valutazione.loc[i, criteri] = punteggi
    valutazione.loc[i, 'Commento'] = commento
print('\nRevisione qualitativa delle nove risposte:')
print(valutazione[['Categoria', 'Modello'] + criteri + ['Commento']].to_string(index=False))
print('\nMedie della rubrica per modello:')
print(valutazione.groupby('Modello', sort=False)[criteri].mean().to_string())
valutazione.to_csv(cartella_risultati / 'valutazione_qualitativa.csv', index=False)
print('Revisione dei risultati osservati; non è uno studio con annotatori umani indipendenti.')


Base, SFT e RLHF: tre copie di DistilGPT-2, senza nuovo training.



Educazione - Base
Prompt: What is reinforcement learning?
Risposta: The answer to this question can be found in the following sections.
The best way for us to learn from our mistakes and give back what we have done as a group, will always involve doing something that people would never do before or after (e



Educazione - SFT
Prompt: What is reinforcement learning?
Risposta: The first thing that I want to say about this topic in the post was how you can learn a lot from experience. The second question, which people have been asking me for quite some time now on my own and what they think of it all



Educazione - RLHF
Prompt: What is reinforcement learning?
Risposta: Learning in real life, not just a theory of how to learn it. It’s the same thing as teaching about motivation and creativity—that's what motivates you for doing that stuff right now…
You can get started by taking part



Coding - Base
Prompt: How do I read a CSV file in Python?
Risposta: Yes, you can download the latest version of SQL Server to see how it works.
The following code uses this same technique for reading JSON files from your web server and using MySQL database as an example output into Excel format (as shown above): Note


...


Customer care - Base
Prompt: How do I track my order?
Risposta: What's the problem with a "best" Order of Duty.
The best orders are those that have no business being paid by their customers and they can't afford to buy them for free, but only when there is something wrong in this market which



Customer care - SFT
Prompt: How do I track my order?
Risposta: What are the items you want to buy for a friend of mine, and what's your favorite item in their cart inventory.
I am currently tracking all purchases from Amazon on Facebook!



Customer care - RLHF
Prompt: How do I track my order?
Risposta: "I can't."
The first thing you need to know is that the following will help with your orders. For example, if a package arrives at me and it doesn´t arrive in one place when they have been sent out by UPS or


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Metriche per risposta:
    Categoria Modello     BLEU  ROUGE-1  BERTScore
   Educazione    Base 0.004754 0.092308   0.835881
   Educazione     SFT 0.005145 0.121212   0.832527
   Educazione    RLHF 0.007094 0.196721   0.830657
       Coding    Base 0.005783 0.207792   0.824712
       Coding     SFT 0.006259 0.181818   0.811733
       Coding    RLHF 0.005653 0.153846   0.808780
Customer care    Base 0.006410 0.187500   0.833048
Customer care     SFT 0.008251 0.192308   0.850933
Customer care    RLHF 0.006568 0.222222   0.836334

Medie per modello:
             BLEU   ROUGE-1  BERTScore
Modello                               
Base     0.005649  0.162533   0.831213
SFT      0.006552  0.165113   0.831731
RLHF     0.006439  0.190930   0.825257

Rubrica: 1 = pessimo, 3 = accettabile, 5 = eccellente
     Criterio                               Descrizione Scala
     Cortesia              Tono educato e professionale   1-5
Informatività               Fornisce informazioni utili   1-5
  Risoluzi

# Esercizio 2 — Valutazione della robustezza e valutazione umana

In base a quanto appreso a lezione, si richiede di valutare la capacità di tre modelli di
classificazione di gestire testi appartenenti a categorie differenti e caratterizzati dalla presenza di rumore linguistico.

#### 1. Valutazione automatica della robustezza

Utilizzando i testi presenti nel percorso `texts/`:

1. Valutare le prestazioni delle tre varianti del modello sui **testi originali/puliti**.
2. Creare, a partire dagli stessi testi, una versione **rumorosa**, introducendo errori ortografici, abbreviazioni e/o slang.
3. Valutare nuovamente le tre varianti del modello sugli stessi esempi della loro versione rumorosa.
4. Confrontare le prestazioni ottenute sui dati puliti e sui dati rumorosi.

Utilizzare come metrica principale la **macro-F1** e, facoltativamente, riportare anche l'accuracy.

Per ciascun modello riportare almeno:

| Modello   | Macro-F1 (pulito) | Macro-F1 (rumoroso) | Δ Macro-F1 |
| --------- | ----------------: | ------------------: | ---------: |
| Base      |                   |                     |            |
| SFT       |                   |                     |            |
| RLHF      |                   |                     |            |

#### 2. Analisi della perplessità per categoria

Calcolare la perplessità (Perplexity) utilizzando il modello GPT-2 separatamente per le diverse categorie presenti nei testi della cartella.

Per ciascuna categoria:

  1. Calcolare la perplessità sui testi disponibili.
  2. Confrontare i valori ottenuti tra le diverse categorie.
  3. Ordinare le categorie in base alla perplessità osservata.
  4. Discutere le possibili ragioni delle differenze, considerando caratteristiche linguistiche e strutturali dei testi.

Infine stampare a video i risultati ottenuti.


In [2]:
### Scrivi la tua soluzione

from sklearn.metrics import f1_score, accuracy_score

categorie = ['news', 'fiction', 'code']
file_categorie = {
    'testo 1.txt': 'news',
    'testo 2.txt': 'fiction',
    'testo 3.txt': 'code'
}
testi = [
    {'File': nome, 'Categoria': categoria,
     'Testo': (Path('texts') / nome).read_text(encoding='utf-8-sig')}
    for nome, categoria in file_categorie.items()
]

# I file forniti sono la baseline, anche se contengono già alcuni errori.
def aggiungi_rumore(testo):
    sostituzioni = [
        ('Ministero', 'Minstero'), ('Secondo', 'Secndo'),
        ('pioggia', 'piogia'), ('mentre', 'mntre'),
        ('sequence', 'seqnce'), ('return', 'retrn'), (' che ', ' ke ')
    ]
    for originale, rumoroso in sostituzioni:
        testo = testo.replace(originale, rumoroso)
    return testo

# Adatto la loss della lezione a una classificazione zero-shot.
# Confronto soltanto la probabilità della categoria dopo il testo.
def classifica_testo(modello, testo):
    prompt = 'Classify the text as news, fiction or code.\nText: ' + testo + '\nCategory:'
    prompt_ids = tokenizer(prompt, return_tensors='pt').input_ids
    nll = {}
    for categoria in categorie:
        categoria_ids = tokenizer(' ' + categoria, return_tensors='pt', add_special_tokens=False).input_ids
        input_ids = torch.cat([prompt_ids, categoria_ids], dim=1)
        labels = input_ids.clone()
        labels[:, :prompt_ids.shape[1]] = -100
        with torch.no_grad():
            loss = modello(input_ids, labels=labels).loss
        nll[categoria] = loss.item() * categoria_ids.shape[1]
    return min(nll, key=nll.get), nll

predizioni = []
metriche_robustezza = []
for nome, modello in modelli.items():
    y_vero = []
    y_pulito = []
    y_rumoroso = []
    for esempio in testi:
        testo_rumoroso = aggiungi_rumore(esempio['Testo'])
        pred_pulito, nll_pulito = classifica_testo(modello, esempio['Testo'])
        pred_rumoroso, nll_rumoroso = classifica_testo(modello, testo_rumoroso)
        y_vero.append(esempio['Categoria'])
        y_pulito.append(pred_pulito)
        y_rumoroso.append(pred_rumoroso)
        predizioni.append({
            'Modello': nome, 'File': esempio['File'], 'Categoria': esempio['Categoria'],
            'Pred_pulito': pred_pulito, 'Pred_rumoroso': pred_rumoroso,
            'Testo_originale': esempio['Testo'], 'Testo_rumoroso': testo_rumoroso,
            'NLL_pulito': nll_pulito, 'NLL_rumoroso': nll_rumoroso
        })
    f1_pulito = f1_score(y_vero, y_pulito, labels=categorie, average='macro', zero_division=0)
    f1_rumoroso = f1_score(y_vero, y_rumoroso, labels=categorie, average='macro', zero_division=0)
    metriche_robustezza.append({
        'Modello': nome, 'Macro-F1 (pulito)': f1_pulito,
        'Macro-F1 (rumoroso)': f1_rumoroso, 'Delta Macro-F1': f1_rumoroso - f1_pulito,
        'Accuracy (pulito)': accuracy_score(y_vero, y_pulito),
        'Accuracy (rumoroso)': accuracy_score(y_vero, y_rumoroso)
    })

df_predizioni = pd.DataFrame(predizioni)
df_robustezza = pd.DataFrame(metriche_robustezza)
print('\nPredizioni sui testi originali e rumorosi:')
print(df_predizioni[['Modello', 'File', 'Categoria', 'Pred_pulito', 'Pred_rumoroso']].to_string(index=False))
print('\nRobustezza (delta = rumoroso - pulito):')
print(df_robustezza.to_string(index=False))
df_predizioni.drop(columns=['NLL_pulito', 'NLL_rumoroso']).to_csv(
    cartella_risultati / 'predizioni.csv', index=False
)
df_robustezza.to_csv(cartella_risultati / 'robustezza.csv', index=False)

def calculate_perplexity(model, tokenizer, text):
    input_ids = tokenizer(text, return_tensors='pt').input_ids
    with torch.no_grad():
        loss = model(input_ids, labels=input_ids).loss
    return torch.exp(loss).item()

perplessita = []
for esempio in testi:
    perplessita.append({
        'Categoria': esempio['Categoria'], 'File': esempio['File'],
        'Perplessità': calculate_perplexity(model_base, tokenizer, esempio['Testo'])
    })
df_perplessita = pd.DataFrame(perplessita).sort_values('Perplessità')
print('\nCategorie ordinate per perplessità crescente:')
print(df_perplessita.to_string(index=False))
df_perplessita.to_csv(cartella_risultati / 'perplessita.csv', index=False)

with open(cartella_risultati / 'robustezza_perplessita.json', 'w', encoding='utf-8') as file:
    json.dump({
        'predizioni': predizioni,
        'robustezza': metriche_robustezza,
        'perplessita': df_perplessita.to_dict('records')
    }, file, ensure_ascii=False, indent=2)



Predizioni sui testi originali e rumorosi:
Modello        File Categoria Pred_pulito Pred_rumoroso
   Base testo 1.txt      news        news          news
   Base testo 2.txt   fiction        news          news
   Base testo 3.txt      code        code          news
    SFT testo 1.txt      news        news          news
    SFT testo 2.txt   fiction        news          news
    SFT testo 3.txt      code        code          news
   RLHF testo 1.txt      news        news          news
   RLHF testo 2.txt   fiction        news          news
   RLHF testo 3.txt      code        code          news

Robustezza (delta = rumoroso - pulito):
Modello  Macro-F1 (pulito)  Macro-F1 (rumoroso)  Delta Macro-F1  Accuracy (pulito)  Accuracy (rumoroso)
   Base           0.555556             0.166667       -0.388889           0.666667             0.333333
    SFT           0.555556             0.166667       -0.388889           0.666667             0.333333
   RLHF           0.555556             0.16


Categorie ordinate per perplessità crescente:
Categoria        File  Perplessità
     code testo 3.txt    11.839002
     news testo 1.txt   247.523636
  fiction testo 2.txt   275.926727


Le tre copie di DistilGPT-2 seguono la simulazione del 23–24 settembre: hanno gli stessi pesi, quindi le differenze di generazione dipendono dal sampling, senza dimostrare effetti di SFT o RLHF. La revisione usa la rubrica della lezione sulle nove risposte effettive. Nessuna risolve il compito: BLEU è circa 0,006 e ROUGE-1 fra 0,163 e 0,191, mentre BERTScore resta circa 0,83. La vicinanza semantica non garantisce una risposta utile.

La classificazione adatta la loss del 25 settembre scegliendo la categoria con minore NLL del completamento. Con gli stessi pesi le predizioni coincidono: macro-F1 passa da 0,556 sui file originali a 0,167 dopo il rumore, con delta −0,389. I file sono conservati anche nei refusi già presenti e il codice viene trattato come testo. Tre esempi permettono soltanto questo confronto dimostrativo.

La perplessità crescente è **code 11,84 → news 247,52 → fiction 275,93**. Il codice contiene strutture e token ricorrenti, mentre i due testi italiani hanno lessico e costruzioni meno prevedibili per un modello addestrato soprattutto in inglese; la narrativa è più varia. Una PPL bassa non rende corretto il codice Fibonacci fornito e non equivale a una buona classificazione.
